# YOLO11-Pose 3 kelas — fine-tuning sebagai hipotesis yang diuji terhadap baseline v2

**Tujuan.** Menguji apakah kepala *pose* (17 keypoint tambahan) memperbaiki deteksi `lying_on_ground` dibanding
baseline v2 (`yolo11n` detect). Hanya kelas lying yang memicu alarm, jadi ukuran utama adalah **lying vs bukan-lying**
(AP dan recall pada FPR tetap), bukan mAP rata-rata.

**Status.** Notebook ini ditulis tanpa GPU. Bagian yang butuh GPU **belum dijalankan**; jangan mengutip angka sebelum
notebook dijalankan di Kaggle. Disusun dari template `yolo11.ipynb` (EXP-007, two-stage detect) dengan urutan bagian
yang sama: (1) install & GPU, (2) deteksi dataset, (3) data.yaml, (4) arsitektur, (5) fine-tuning, (6) evaluasi.

**Yang perlu diingat sebelum membaca hasil**
- Dataset **tidak punya keypoint asli**. Keypoint dibuat oleh model *teacher* (pseudo-label), jadi metrik pose (OKS)
  hanya "kesesuaian dengan teacher", bukan kebenaran. Metrik utama memakai box + kelas.
- Kepala kelas pada model pose sama dengan model detect: keypoint **tidak otomatis** menyaring kelas. Klaim bahwa pose
  menolak alarm palsu kasur/sofa adalah hipotesis yang diuji di sini, bukan alasan.
- Lying di test hanya ada di **6 video**. Semua angka lying memakai CI bootstrap per video dan banyak yang akan
  berlabel *tidak konklusif*. Itu jujur, bukan kegagalan notebook.
- Metrik per frame dari 6 video tidak sama dengan kinerja per kejadian (>10 detik). Notebook ini tidak pernah menyimpulkan
  "layak deploy".

## Pengaturan
Semua angka yang boleh diubah ada di sel ini. Variabel lingkungan `POSE_WORK` dan `POSE_DEBUG_SUBSET` hanya untuk uji
kering di mesin tanpa GPU (membatasi jumlah gambar); untuk hasil sungguhan biarkan kosong.

In [ ]:
import os
from pathlib import Path

RUN_ID_START = 12          # nomor awal penamaan folder run; BUKAN nomor EXP resmi (menunggu persetujuan tim)
TAG = "pose"               # awalan nama folder run
SEED = 0
FORCE_RECOMPUTE = False    # True: hitung ulang pseudo-label, training, dan skor walau hasilnya sudah ada

# Uji kering: batasi gambar per split (None = semua). Hanya untuk mencoba alur, hasilnya tidak bermakna.
DEBUG_SUBSET = int(os.environ["POSE_DEBUG_SUBSET"]) if os.environ.get("POSE_DEBUG_SUBSET") else None

# Saklar bagian mahal
RUN_PSEUDO_LABEL = True
RUN_EXPLORATION = True
RUN_FINETUNE = True        # bagian 5: one-stage penuh vs two-stage
RUN_SEEDS = True           # 3 seed untuk satu kandidat akhir
RUN_ROBUSTNESS = True
RUN_TRAIN_SPLIT_VAL = False  # mAP di split train (hanya informasi; makan waktu)
ALLOW_CPU_TRAIN = False    # tanpa GPU, training dilewati
FINAL_WEIGHTS_OVERRIDE = None  # isi path best.pt untuk langsung mengevaluasi bobot lain (bagian 6) tanpa training
FINAL_IMGSZ_OVERRIDE = 640

# Teacher untuk pseudo-label keypoint
TEACHER = "yolo11m-pose.pt"  # alternatif "yolo11x-pose.pt": lebih teliti, lebih lambat
TEACHER_IMGSZ, TEACHER_CONF = 960, 0.25
MATCH_IOU, MATCH_CONF = 0.5, 0.3   # match sah: IoU >= 0,5 terhadap box label DAN conf person >= 0,3
KP_V2, KP_V1 = 0.5, 0.3            # visibilitas keypoint: v=2 bila conf >= 0,5; v=1 bila >= 0,3; selain itu v=0

# Resep baseline v2 (fakta tim): yolo11n detect, AdamW lr0 0,001, cos_lr, cls 1,0, patience 30, 70 epoch, batch 16,
# imgsz 640, seed 0, augmentasi bawaan. AUG di bawah = nilai bawaan ultralytics, ditulis agar terlihat.
BASE_HP = dict(epochs=70, patience=30, batch=16, imgsz=640, optimizer="AdamW", lr0=0.001, cos_lr=True,
               cls=1.0, seed=SEED, deterministic=True)
AUG = dict(hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, translate=0.1, scale=0.5, fliplr=0.5, flipud=0.0,
           degrees=0.0, mosaic=1.0)

# Eksplorasi (bagian 4b): maks 6 run, maks 30 epoch/run. Satu perubahan per run dari R1.
EXPLORE_EPOCHS, EXPLORE_PATIENCE = 30, 10
EXPLORE_RUNS = [
    dict(rid="R0", model="yolo11n.pt", note="kontrol detect: resep baseline, anggaran epoch sama"),
    dict(rid="R1", model="yolo11n-pose.pt", note="pose, resep identik baseline (kecuali anggaran epoch)"),
    dict(rid="R2", model="yolo11s-pose.pt", note="lebih dalam/lebar (varian s)"),
    dict(rid="R3", model="yolo11n-pose.pt", freeze=10, note="10 layer awal dibekukan"),
    dict(rid="R4", model="yolo11n-pose.pt", imgsz=800, note="resolusi 800"),
    dict(rid="R5", model="yolo11n-pose.pt", optimizer="SGD", lr0=0.01, note="optimizer SGD dengan lr0 bawaannya"),
]

# Bagian 5
FINAL_EPOCHS, FINAL_PATIENCE = 70, 30
STAGE1_EPOCHS, STAGE1_FREEZE = 20, 10
STAGE2_EPOCHS, STAGE2_PATIENCE, STAGE2_LR_FACTOR = 50, 20, 0.1
SEEDS = [0, 1, 2]

# Bagian 6
TARGET_FPRS = (0.01, 0.05)          # ambang dipilih di VAL agar FPR frame-normal <= nilai ini
REPORT_FPRS = (0.01, 0.03, 0.05, 0.10)
N_BOOT = 1000
SCORE_CONF = 0.05                   # sama dengan eval_lying.predict_scores

# Berkas repo (eval_lying.py, skor baseline) diunduh dari commit terkunci bila tidak ada di /kaggle/input
REPO_COMMIT = "0bd37627efcb1ce3f4e6cd5ad3024817e76ef437"
RAW_BASE = f"https://raw.githubusercontent.com/0Raja0-RA/nursing-home-fall-detection/{REPO_COMMIT}"

KAGGLE = Path("/kaggle/working").exists()
WORK = Path(os.environ.get("POSE_WORK", "/kaggle/working" if KAGGLE else "runs/pose_work")).resolve()
WORK.mkdir(parents=True, exist_ok=True)
DIAG = WORK / "diag"
DIAG.mkdir(exist_ok=True)
# Folder tambahan untuk mencari dataset (opsional; pisahkan dengan os.pathsep), mis. dataset lokal
EXTRA_SEARCH = [Path(p) for p in os.environ.get("POSE_DATA_SEARCH", "").split(os.pathsep) if p]
SPLITS = ("train", "val", "test")
print(f"Kaggle={KAGGLE} | WORK={WORK} | DEBUG_SUBSET={DEBUG_SUBSET}")

## 1. Install Ultralytics & verifikasi GPU
**1a.** Memasang ultralytics (dikunci ke versi yang dipakai saat notebook disusun dan diperiksa) dan mencetak versi serta
GPU. Di Kaggle, GPU wajib (Settings → Accelerator → GPU T4).

In [ ]:
import subprocess
import sys

ULTRALYTICS_VERSION = "8.4.171"   # versi yang kodenya diperiksa saat menyusun notebook
if KAGGLE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"ultralytics=={ULTRALYTICS_VERSION}"], check=True)

import torch
import ultralytics
from ultralytics import YOLO

HAS_GPU = torch.cuda.is_available()
DEVICE = 0 if HAS_GPU else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if HAS_GPU else "tidak ada"
print("ultralytics:", ultralytics.__version__, "| torch:", torch.__version__, "| GPU:", GPU_NAME)
if ultralytics.__version__ != ULTRALYTICS_VERSION:
    print(f"⚠ versi ultralytics berbeda dari {ULTRALYTICS_VERSION}; nama atribut API bisa berubah")
if KAGGLE:
    assert HAS_GPU, "GPU tidak aktif. Kaggle: Settings -> Accelerator -> GPU T4 x1, lalu jalankan ulang."
elif not HAS_GPU:
    print("Tanpa GPU: training dan evaluasi berbasis model akan dilewati (ALLOW_CPU_TRAIN=False).")

**1b. Fungsi bantu murni.** Tidak bergantung pada GPU maupun ultralytics, sehingga bisa diuji di mesin mana pun:
ambang FPR, AP, ECE, bootstrap per video, penyusunan baris label 56 kolom, gangguan gambar untuk uji ketahanan, dan
`interpret()`. `ap_lying` sengaja sama persis dengan `eval_lying.average_precision`; kesamaannya di-`assert` di 1d.

In [ ]:
import math
import re

import numpy as np

GT_NORMAL, GT_TRANS, GT_LYING = 0, 1, 2
CLASS_NAMES = {0: "normal", 1: "transitional", 2: "lying_on_ground"}
FLIP_IDX = [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15]   # urutan keypoint COCO bila gambar dibalik
COCO_SKELETON = [(15, 13), (13, 11), (16, 14), (14, 12), (11, 12), (5, 11), (6, 12), (5, 6), (5, 7),
                 (6, 8), (7, 9), (8, 10), (1, 2), (0, 1), (0, 2), (1, 3), (2, 4), (3, 5), (4, 6)]
WARN_STD = "⚠ bukan standar baku; belum tervalidasi untuk dataset ini"


def video_of(stem):
    """Kelompok kejadian, sama dengan eval_lying.video_of: satu skenario MCF (8 kamera) = satu video."""
    m = re.match(r"mcf_s(\d+)_c\d_f\d+$", stem)
    if m:
        return f"mcf-s{m.group(1)}"
    m = re.match(r"urfall_((?:fall|adl)-\d+)_f\d+$", stem)
    if m:
        return f"urfall-{m.group(1)}"
    return stem


def source_of(stem):
    return "MCF" if stem.startswith("mcf_") else ("UR Fall" if stem.startswith("urfall_") else "lain")


def xywhn_to_xyxy(b):
    cx, cy, w, h = b
    return (cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2)


def iou_xyxy(a, b):
    iw = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    ih = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = iw * ih
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def kp_values(kxy, kconf, v2=0.5, v1=0.3):
    """17 keypoint teacher -> 51 nilai (x, y, v). Di luar gambar atau conf rendah -> v=0 dengan x=y=0."""
    out = []
    for (x, y), c in zip(kxy, kconf, strict=True):
        v = 2 if c >= v2 else (1 if c >= v1 else 0)
        if v and not (0.0 <= x <= 1.0 and 0.0 <= y <= 1.0):
            v = 0
        out += [float(x), float(y), v] if v else [0.0, 0.0, 0]
    return out


def build_pose_row(orig_line, kxy=None, kconf=None, v2=0.5, v1=0.3):
    """Baris label 56 kolom: kelas + box ASLI (string apa adanya) + 17 keypoint. Tanpa match -> semua v=0."""
    parts = orig_line.split()
    if len(parts) < 5:
        raise ValueError(f"baris label harus >= 5 kolom: {orig_line!r}")
    vals = [0.0, 0.0, 0] * 17 if kxy is None else kp_values(kxy, kconf, v2, v1)
    kp = []
    for i in range(17):
        x, y, v = vals[3 * i:3 * i + 3]
        kp += [f"{x:.6f}", f"{y:.6f}", str(int(v))]
    return " ".join(parts[:5] + kp)


def n_visible(row):
    v = row.split()[5:][2::3]
    return sum(int(float(x)) >= 1 for x in v)


def ap_lying(pos, neg):
    """AP lying-vs-normal; identik dengan eval_lying.average_precision (skor sama = satu ambang)."""
    pos, neg = np.asarray(pos, float), np.asarray(neg, float)
    if len(pos) == 0 or len(neg) == 0:
        return float("nan")
    ap, prev = 0.0, 0.0
    for t in np.unique(np.concatenate([pos, neg]))[::-1]:
        tp, fp = int((pos >= t).sum()), int((neg >= t).sum())
        r = tp / len(pos)
        ap += (r - prev) * tp / (tp + fp)
        prev = r
    return float(ap)


def thr_for_fpr(neg, fpr):
    """Ambang terkecil sehingga paling banyak floor(fpr*n) frame normal punya skor >= ambang."""
    ns = np.sort(np.asarray(neg, float))[::-1]
    k = int(np.floor(fpr * len(ns)))
    if len(ns) == 0 or k >= len(ns):
        return 0.0
    return float(np.nextafter(ns[k], np.inf))


def rates_at(scores, gt, thr):
    scores, gt = np.asarray(scores, float), np.asarray(gt, int)
    pos, neg, tr = scores[gt == GT_LYING], scores[gt == GT_NORMAL], scores[gt == GT_TRANS]
    tp, fp = int((pos >= thr).sum()), int((neg >= thr).sum())
    return dict(recall=tp / len(pos) if len(pos) else float("nan"),
                precision=tp / (tp + fp) if tp + fp else float("nan"),
                fp_normal=fp, fpr=fp / len(neg) if len(neg) else float("nan"),
                trans_as_lying=float((tr >= thr).mean()) if len(tr) else float("nan"))


def recall_at_fpr(scores, gt, fpr):
    """Recall lying pada FPR frame-normal tertentu; ambang dihitung dari data yang sama (deskriptif)."""
    scores, gt = np.asarray(scores, float), np.asarray(gt, int)
    thr = thr_for_fpr(scores[gt == GT_NORMAL], fpr)
    return rates_at(scores, gt, thr)["recall"], thr


def ece_binary(prob, y, n_bins=10):
    """Expected Calibration Error + tabel reliability (10 bin sama lebar di [0,1])."""
    prob, y = np.asarray(prob, float), np.asarray(y, int)
    edges = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(prob, edges[1:-1], right=False), 0, n_bins - 1)
    rows, ece = [], 0.0
    for b in range(n_bins):
        m = idx == b
        if m.any():
            conf, acc = float(prob[m].mean()), float(y[m].mean())
            ece += m.mean() * abs(conf - acc)
            rows.append((edges[b], edges[b + 1], int(m.sum()), conf, acc))
        else:
            rows.append((edges[b], edges[b + 1], 0, float("nan"), float("nan")))
    return float(ece), rows


def _video_index(videos):
    by = {}
    for i, v in enumerate(videos):
        by.setdefault(v, []).append(i)
    keys = sorted(by)
    return keys, [np.array(by[k]) for k in keys]


def bootstrap_by_video(fn, scores, gt, videos, n_boot=1000, seed=0):
    """CI 95% dengan resampling VIDEO (frame dalam satu kejadian berkorelasi). fn(scores, gt) -> float."""
    scores, gt = np.asarray(scores, float), np.asarray(gt, int)
    keys, groups = _video_index(list(videos))
    rng = np.random.default_rng(seed)
    est, vals = fn(scores, gt), []
    for _ in range(n_boot):
        idx = np.concatenate([groups[k] for k in rng.integers(0, len(keys), len(keys))])
        vals.append(fn(scores[idx], gt[idx]))
    vals = np.asarray(vals, float)
    lo, hi = (np.nanpercentile(vals, 2.5), np.nanpercentile(vals, 97.5)) if np.isfinite(vals).any() else (np.nan, np.nan)
    return float(est), float(lo), float(hi)


def paired_bootstrap(fn, s_base, s_cand, gt, videos, n_boot=1000, seed=0):
    """Selisih berpasangan (kandidat - baseline) dengan video yang SAMA tiap resampling."""
    s_base, s_cand, gt = np.asarray(s_base, float), np.asarray(s_cand, float), np.asarray(gt, int)
    keys, groups = _video_index(list(videos))
    rng = np.random.default_rng(seed)
    d = []
    for _ in range(n_boot):
        idx = np.concatenate([groups[k] for k in rng.integers(0, len(keys), len(keys))])
        d.append(fn(s_cand[idx], gt[idx]) - fn(s_base[idx], gt[idx]))
    d = np.asarray(d, float)
    d = d[np.isfinite(d)]
    delta = fn(s_cand, gt) - fn(s_base, gt)
    return dict(delta=float(delta), lo=float(np.percentile(d, 2.5)), hi=float(np.percentile(d, 97.5)),
                p=float((d > 0).mean()), n_boot=int(len(d)))


def ap_fn(scores, gt):
    return ap_lying(scores[gt == GT_LYING], scores[gt == GT_NORMAL])


def make_recall_fpr_fn(fpr):
    return lambda scores, gt: recall_at_fpr(scores, gt, fpr)[0]


# ---- gangguan gambar untuk uji ketahanan (tanpa flip vertikal) ----
def perturb(img, kind, seed=0):
    import cv2
    rng = np.random.default_rng(seed)
    h, w = img.shape[:2]
    f = img.astype(np.float32)
    if kind == "bright+30":
        out = f * 1.3
    elif kind == "bright-30":
        out = f * 0.7
    elif kind == "blur":
        out = cv2.GaussianBlur(img, (7, 7), 2.0).astype(np.float32)
    elif kind == "noise":
        out = f + rng.normal(0, 15, f.shape)
    elif kind == "jpeg30":
        ok, buf = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 30])
        out = cv2.imdecode(buf, cv2.IMREAD_COLOR).astype(np.float32)
    elif kind == "scale0.75":   # orang tampak lebih kecil/jauh: gambar diperkecil di kanvas abu-abu
        small = cv2.resize(img, (int(w * 0.75), int(h * 0.75)), interpolation=cv2.INTER_AREA)
        out = np.full_like(img, 114)
        y0, x0 = (h - small.shape[0]) // 2, (w - small.shape[1]) // 2
        out[y0:y0 + small.shape[0], x0:x0 + small.shape[1]] = small
        out = out.astype(np.float32)
    elif kind == "scale1.25":   # zoom tengah
        ch, cw = int(h / 1.25), int(w / 1.25)
        y0, x0 = (h - ch) // 2, (w - cw) // 2
        out = cv2.resize(img[y0:y0 + ch, x0:x0 + cw], (w, h), interpolation=cv2.INTER_LINEAR).astype(np.float32)
    elif kind in ("rot+10", "rot-10"):
        ang = 10 if kind == "rot+10" else -10
        M = cv2.getRotationMatrix2D((w / 2, h / 2), ang, 1.0)
        out = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT_101).astype(np.float32)
    elif kind == "occlude20":   # penutup persegi acak seluas 20% gambar
        area = 0.20 * h * w
        ar = rng.uniform(0.5, 2.0)
        rh, rw = int(min(h, math.sqrt(area / ar))), int(min(w, math.sqrt(area * ar)))
        y0, x0 = int(rng.integers(0, h - rh + 1)), int(rng.integers(0, w - rw + 1))
        out = f.copy()
        out[y0:y0 + rh, x0:x0 + rw] = 114
    else:
        raise ValueError(kind)
    return np.clip(out, 0, 255).astype(np.uint8)


PERTURBATIONS = ["bright+30", "bright-30", "blur", "noise", "jpeg30", "scale0.75", "scale1.25",
                 "rot+10", "rot-10", "occlude20"]


# ---- interpretasi otomatis ----
def interpret(name, value, n_videos=None, ci=None):
    """-> (label, saran). Ambang = target tim (EXPERIMENTS.md) atau heuristik umum; bukan standar baku."""
    if (n_videos is not None and n_videos < 10) or (ci is not None and (ci[1] - ci[0]) > 0.30):
        return ("TIDAK KONKLUSIF (n kecil)",
                f"angka ditampilkan tanpa label (n_video={n_videos}, CI={None if ci is None else tuple(round(c, 3) for c in ci)}); butuh lebih banyak video")
    if isinstance(value, (int, float)) and math.isnan(value):
        return "TIDAK TERDEFINISI", "metrik tidak bisa dihitung (mis. tidak ada prediksi positif atau tidak ada sampel)"
    if name == "recall_lying":
        if value >= 0.92:
            return "sangat baik (target ideal tim)", "pertahankan; cek alarm palsu pada titik kerja yang sama"
        if value >= 0.85:
            return "memenuhi minimum (target tim)", "naikkan bila alarm palsu masih terkendali"
        if value >= 0.70:
            return "kurang optimal (risiko jatuh terlewat)", "periksa video lying yang gagal; data/ambang"
        return "tidak layak", "recall terlalu rendah untuk alarm keselamatan"
    if name == "precision":
        if value >= 0.80:
            return "sangat baik (target ideal tim)", "-"
        if value >= 0.65:
            return "memenuhi (target tim)", "-"
        if value >= 0.50:
            return "banyak alarm palsu", "periksa video penyumbang alarm palsu"
        return "buruk", "alarm palsu dominan"
    if name == "map50":
        if value >= 0.85:
            return "sangat baik (target ideal tim)", "-"
        if value >= 0.75:
            return "memenuhi (target tim)", "-"
        if value >= 0.60:
            return "sedang", "lihat AP per kelas; lying yang menentukan alarm"
        return "lemah", "lihat AP per kelas; lying yang menentukan alarm"
    if name == "latency_ms":
        if value <= 10:
            return "ideal (target tim)", "catat GPU pengukur; perangkat tujuan belum jelas"
        if value <= 30:
            return "memenuhi (target tim)", "catat GPU pengukur; perangkat tujuan belum jelas"
        return "tidak real-time", "ukur ulang di perangkat tujuan"
    if name == "gap_val_test":      # dalam poin persen
        if value <= 5:
            return "stabil (heuristik)", "-"
        if value <= 15:
            return "pergeseran sedang antar-skenario (heuristik)", "lihat per video"
        return "pergeseran besar (heuristik)", "skenario test berbeda dari val; lihat per video"
    if name == "robust_drop":       # penurunan relatif (0..1) pada gangguan terburuk
        if value <= 0.10:
            return "tahan (heuristik)", "-"
        if value <= 0.25:
            return "sedang (heuristik)", "pertimbangkan augmentasi sesuai gangguan terburuk"
        return "rapuh (heuristik)", "augmentasi/data untuk gangguan terburuk"
    if name == "ece":
        if value <= 0.05:
            return "baik (heuristik)", "-"
        if value <= 0.15:
            return "sedang (heuristik)", "confidence YOLO bukan probabilitas; jangan dibaca sebagai peluang"
        return "buruk (heuristik)", "kalibrasi (mis. temperature scaling di val) sebelum dipakai sebagai peluang"
    if name == "fa_concentration":  # value: {video: jumlah alarm palsu}
        total = sum(value.values())
        if total == 0:
            return "tidak ada alarm palsu", "-"
        top, n = max(value.items(), key=lambda kv: kv[1])
        if n / total > 0.5:
            return f"TERKONSENTRASI di {top}", "periksa konteks (mis. sofa/matras); menaikkan ambang tidak otomatis menolong"
        return "tersebar", "-"
    if name == "collapse":          # value: {"total": recall, "per_video": {video: recall}}
        tot = value["total"]
        bad = [v for v, r in sorted(value["per_video"].items()) if r < 0.5 * tot or r < 0.20]
        if bad:
            return f"RUNTUH di {', '.join(bad)}", "periksa frame video tersebut"
        return "tidak ada skenario runtuh", "-"
    if name == "paired":            # value: {"delta","lo","hi","p"}
        if value["delta"] > 0 and value["lo"] > 0 and value["p"] >= 0.9:
            return "LEBIH BAIK", "-"
        if value["p"] <= 0.1:
            return "LEBIH BURUK", "-"
        return "TIDAK KONKLUSIF", "selisih dalam noise; pilih model lebih sederhana"
    if name == "s23_fa":            # value: {"baseline": n, "candidate": n}
        if value["candidate"] < value["baseline"]:
            return "alarm palsu s23 turun", "-"
        return "alarm palsu s23 TIDAK turun", "hipotesis 'pose menolak sofa/matras' tidak didukung"
    raise ValueError(f"metrik tidak dikenal: {name}")


VERDICTS = ["KANDIDAT LAYAK DIUJI LANJUT", "TIDAK KONKLUSIF", "PERLU PERBAIKAN"]   # urut dari terbaik ke terburuk


def component_class(name, label):
    if label.startswith(("TIDAK KONKLUSIF", "TIDAK TERDEFINISI")):
        return "TIDAK KONKLUSIF"
    bad = {"recall_lying": ("kurang optimal", "tidak layak"), "collapse": ("RUNTUH",),
           "s23_fa": ("alarm palsu s23 TIDAK",), "paired": ("LEBIH BURUK",)}
    return "PERLU PERBAIKAN" if any(label.startswith(b) for b in bad.get(name, ())) else "KANDIDAT LAYAK DIUJI LANJUT"


def final_verdict(labels):
    """labels: {nama_komponen: label}. Verdict = yang TERBURUK. Tidak pernah 'layak deploy'."""
    return max((component_class(k, v) for k, v in labels.items()), key=VERDICTS.index)


def show_interp(name, value, **kw):
    label, saran = interpret(name, value, **kw)
    shown = value if not isinstance(value, float) else round(value, 4)
    print(f"[{name}] {shown} -> {label} | saran: {saran}  ({WARN_STD})")
    return label

**1c. Uji unit fungsi murni.** Data sintetis dengan jawaban yang diketahui; mencakup setiap cabang `interpret()`
termasuk penjaga n kecil. Bila ada `assert` gagal, berhenti di sini.

In [ ]:
def run_unit_tests():
    n = 0

    def check(cond, msg):
        nonlocal n
        assert cond, msg
        n += 1

    # IoU dan konversi box
    check(abs(iou_xyxy((0, 0, 1, 1), (0, 0, 1, 1)) - 1) < 1e-9, "IoU identik")
    check(iou_xyxy((0, 0, 1, 1), (2, 2, 3, 3)) == 0, "IoU terpisah")
    check(abs(iou_xyxy(xywhn_to_xyxy((0.5, 0.5, 0.2, 0.2)), (0.4, 0.4, 0.6, 0.6)) - 1) < 1e-9, "xywhn->xyxy")
    # video dan sumber
    check(video_of("mcf_s19_c3_f00123") == "mcf-s19", "video MCF")
    check(video_of("urfall_fall-26_f0042") == "urfall-fall-26", "video UR Fall")
    check(source_of("urfall_adl-10_f0001") == "UR Fall", "sumber")
    # baris label 56 kolom
    orig = "2 0.512345 0.400000 0.300000 0.250000"
    kxy = np.array([[0.1 + 0.01 * i, 0.2] for i in range(17)])
    kc = np.array([0.9] * 5 + [0.4] * 5 + [0.1] * 7)
    row = build_pose_row(orig, kxy, kc)
    p = row.split()
    check(len(p) == 56, "56 kolom")
    check(p[:5] == orig.split(), "kelas + box asli dipertahankan apa adanya")
    vs = [int(x) for x in p[5:][2::3]]
    check(vs == [2] * 5 + [1] * 5 + [0] * 7, "aturan visibilitas v=2/1/0")
    check(p[5 + 3 * 16:5 + 3 * 17] == ["0.000000", "0.000000", "0"], "v=0 -> x=y=0")
    out = kxy.copy(); out[0] = [1.3, 0.5]
    check(build_pose_row(orig, out, np.ones(17)).split()[5:8] == ["0.000000", "0.000000", "0"], "di luar gambar -> v=0")
    nm = build_pose_row(orig)
    check(len(nm.split()) == 56 and n_visible(nm) == 0, "tanpa match: 56 kolom, semua v=0")
    check(n_visible(row) == 10, "hitung keypoint terlihat")
    # ambang FPR dengan banyak skor nol (ties)
    neg = np.array([0.0] * 90 + [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99])
    t = thr_for_fpr(neg, 0.05)
    check((neg >= t).mean() <= 0.05 and (neg >= 0.6).mean() > 0.05, "ambang FPR 5%: ketat, tidak berlebihan")
    check(thr_for_fpr(neg, 0.5) > 0, "FPR 50% dengan 90% nol: ambang tetap > 0")
    scores = np.concatenate([neg, [0.97, 0.96, 0.1, 0.0]])
    gt = np.array([0] * 100 + [2] * 4)
    r, _ = recall_at_fpr(scores, gt, 0.05)
    check(abs(r - 0.5) < 1e-9, "recall@FPR5% = 2/4")
    # AP
    check(ap_lying([0.9, 0.8], [0.1, 0.2]) == 1.0, "AP sempurna")
    check(abs(ap_lying([0.5], [0.9]) - 0.5) < 1e-9, "AP satu negatif di atas")
    check(math.isnan(ap_lying([], [0.1])), "AP tanpa positif = nan")
    # ECE
    rng = np.random.default_rng(0)
    pr = rng.uniform(0, 1, 20000)
    y = (rng.uniform(0, 1, 20000) < pr).astype(int)
    check(ece_binary(pr, y)[0] < 0.02, "ECE terkalibrasi ~0")
    check(ece_binary(np.full(1000, 0.95), np.zeros(1000, int))[0] > 0.9, "ECE terlalu yakin")
    # bootstrap berpasangan
    vids = [f"v{i // 10}" for i in range(200)]
    g2 = np.array([2 if i % 2 else 0 for i in range(200)])
    base = rng.uniform(0, 1, 200)
    cand = np.where(g2 == 2, np.minimum(base + 0.5, 1.0), base)
    pb = paired_bootstrap(ap_fn, base, cand, g2, vids, n_boot=200, seed=0)
    check(pb["delta"] > 0 and pb["p"] > 0.9, "berpasangan: kandidat jelas lebih baik")
    est, lo, hi = bootstrap_by_video(ap_fn, cand, g2, vids, n_boot=200)
    check(lo <= est <= hi, "CI memuat estimasi")
    # gangguan gambar
    img = rng.integers(0, 255, (48, 64, 3), dtype=np.uint8)
    for k in PERTURBATIONS:
        o = perturb(img, k, seed=1)
        check(o.shape == img.shape and o.dtype == np.uint8, f"gangguan {k}")
    check(np.array_equal(perturb(img, "occlude20", 3), perturb(img, "occlude20", 3)), "oklusi deterministik per seed")
    # interpret: setiap cabang
    cases = [
        ("recall_lying", 0.95, "sangat baik"), ("recall_lying", 0.88, "memenuhi minimum"),
        ("recall_lying", 0.75, "kurang optimal"), ("recall_lying", 0.5, "tidak layak"),
        ("precision", 0.85, "sangat baik"), ("precision", 0.70, "memenuhi"),
        ("precision", 0.55, "banyak alarm palsu"), ("precision", 0.3, "buruk"),
        ("map50", 0.9, "sangat baik"), ("map50", 0.8, "memenuhi"), ("map50", 0.65, "sedang"), ("map50", 0.4, "lemah"),
        ("latency_ms", 5, "ideal"), ("latency_ms", 20, "memenuhi"), ("latency_ms", 50, "tidak real-time"),
        ("gap_val_test", 3, "stabil"), ("gap_val_test", 10, "pergeseran sedang"), ("gap_val_test", 20, "pergeseran besar"),
        ("robust_drop", 0.05, "tahan"), ("robust_drop", 0.2, "sedang"), ("robust_drop", 0.4, "rapuh"),
        ("ece", 0.03, "baik"), ("ece", 0.1, "sedang"), ("ece", 0.3, "buruk"),
        ("fa_concentration", {"mcf-s23": 80, "mcf-s19": 3}, "TERKONSENTRASI di mcf-s23"),
        ("fa_concentration", {"a": 5, "b": 5, "c": 5}, "tersebar"), ("fa_concentration", {"a": 0}, "tidak ada"),
        ("collapse", {"total": 0.6, "per_video": {"s19": 0.1, "s21": 0.7}}, "RUNTUH di s19"),
        ("collapse", {"total": 0.6, "per_video": {"s19": 0.55, "s21": 0.7}}, "tidak ada skenario runtuh"),
        ("paired", {"delta": 0.05, "lo": 0.01, "hi": 0.09, "p": 0.97}, "LEBIH BAIK"),
        ("paired", {"delta": -0.05, "lo": -0.09, "hi": -0.01, "p": 0.03}, "LEBIH BURUK"),
        ("paired", {"delta": 0.01, "lo": -0.03, "hi": 0.05, "p": 0.6}, "TIDAK KONKLUSIF"),
        ("s23_fa", {"baseline": 83, "candidate": 40}, "alarm palsu s23 turun"),
        ("s23_fa", {"baseline": 83, "candidate": 90}, "alarm palsu s23 TIDAK turun"),
    ]
    for name, val, expect in cases:
        lab = interpret(name, val)[0]
        check(lab.startswith(expect), f"{name}={val}: '{lab}' bukan '{expect}'")
    # penjaga n kecil mengalahkan label lain
    check(interpret("recall_lying", 0.99, n_videos=6)[0] == "TIDAK KONKLUSIF (n kecil)", "penjaga n_video < 10")
    check(interpret("recall_lying", 0.99, n_videos=12, ci=(0.5, 0.95))[0] == "TIDAK KONKLUSIF (n kecil)", "penjaga CI > 0,30")
    check(interpret("recall_lying", 0.99, n_videos=12, ci=(0.9, 1.0))[0].startswith("sangat baik"), "penjaga tidak aktif")
    check(interpret("precision", float("nan"), n_videos=12)[0] == "TIDAK TERDEFINISI", "NaN tidak boleh berlabel buruk")
    check(final_verdict({"recall_lying": "TIDAK TERDEFINISI"}) == "TIDAK KONKLUSIF", "NaN -> tidak konklusif")
    try:
        interpret("tidak_ada", 1)
        check(False, "nama tak dikenal harus error")
    except ValueError:
        check(True, "-")
    # verdict = yang terburuk
    check(final_verdict({"paired": "LEBIH BAIK", "collapse": "tidak ada skenario runtuh"}) == "KANDIDAT LAYAK DIUJI LANJUT", "verdict baik")
    check(final_verdict({"paired": "LEBIH BAIK", "recall_lying": "TIDAK KONKLUSIF (n kecil)"}) == "TIDAK KONKLUSIF", "verdict tak konklusif")
    check(final_verdict({"paired": "TIDAK KONKLUSIF", "collapse": "RUNTUH di mcf-s19"}) == "PERLU PERBAIKAN", "verdict terburuk")
    check(all("deploy" not in v.lower() for v in VERDICTS), "tidak ada 'layak deploy'")
    return n


print(f"Semua uji unit lulus: {run_unit_tests()} assert")

**1d. Memuat `eval_lying.py` dan skor baseline.** Urutan pencarian: berkas di `/kaggle/input` (bila repo diunggah
sebagai dataset) → repo lokal → unduh dari GitHub pada commit terkunci (`REPO_COMMIT`). `eval_lying.py` dipakai apa adanya:
`predict_scores` (skor gambar = confidence kotak lying tertinggi, 0 bila tidak ada) dan `evaluate`.
Catatan: skrip itu **tidak** punya opsi `--split`, `--thr`, maupun `--compare`; fungsi tersebut dibuat sebagai adaptor
di notebook ini tanpa mengubah skrip.

In [ ]:
import importlib.util
import urllib.request

import pandas as pd


def find_repo_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "ml" / "scripts").is_dir():
            return p
    return None


REPO_ROOT = find_repo_root()


def locate(filename, repo_rel):
    if Path("/kaggle/input").exists():
        for d, _, files in os.walk("/kaggle/input"):
            if filename in files:
                return Path(d) / filename, "kaggle input"
    if REPO_ROOT and (REPO_ROOT / repo_rel).is_file():
        return REPO_ROOT / repo_rel, "repo lokal"
    dst = WORK / "tools" / filename
    if not dst.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f"{RAW_BASE}/{repo_rel}", dst)
    return dst, f"GitHub @ {REPO_COMMIT[:8]}"


el_path, el_src = locate("eval_lying.py", "ml/scripts/eval_lying.py")
spec = importlib.util.spec_from_file_location("eval_lying", el_path)
eval_lying = importlib.util.module_from_spec(spec)
sys.dont_write_bytecode = True
spec.loader.exec_module(eval_lying)
print(f"eval_lying.py: {el_path} ({el_src})")

base_path, base_src = locate("eval_3cls.scores.csv", "ml/data/eval_3cls.scores.csv")
BASE = pd.read_csv(base_path)
assert list(BASE.columns) == ["stem", "gt", "video", "score"], BASE.columns
print(f"skor baseline v2: {base_path} ({base_src}) | {len(BASE)} gambar test")

# Konsistensi: AP versi notebook == AP versi eval_lying pada data nyata
_s, _g = BASE["score"].to_numpy(), BASE["gt"].to_numpy()
assert abs(ap_lying(_s[_g == 2], _s[_g == 0]) - eval_lying.average_precision(_s[_g == 2], _s[_g == 0])) < 1e-12
assert all(video_of(s) == eval_lying.video_of(s) for s in BASE["stem"].head(500))
print(f"baseline v2: AP lying-vs-normal = {ap_lying(_s[_g == 2], _s[_g == 0]):.3f} | "
      + " | ".join(f"recall@FPR{int(f * 100)}% = {recall_at_fpr(_s, _g, f)[0]:.3f}" for f in REPORT_FPRS))

## 2. Deteksi dataset (path fleksibel)
Mencari semua folder yang punya `images/{train,val,test}` dan `labels/{train,val,test}` di `/kaggle/input` (atau
`ml/data/processed*` bila lokal). Pemilihan: `nc == 3`, lalu nama folder memuat `3cls`; bila masih lebih dari satu,
berhenti dan sebutkan kandidatnya (tidak menebak).

In [ ]:
import yaml


def is_dataset_root(d):
    return all((d / k / s).is_dir() for k in ("images", "labels") for s in SPLITS)


def read_nc_names(d):
    for y in sorted(d.glob("*.yaml")):
        try:
            cfg = yaml.safe_load(y.read_text())
            if isinstance(cfg, dict) and "nc" in cfg:
                return int(cfg["nc"]), cfg.get("names")
        except Exception:
            pass
    classes = set()   # tanpa yaml: simpulkan dari maks 300 label
    for f in list((d / "labels" / "train").glob("*.txt"))[:300]:
        classes |= {int(line.split()[0]) for line in f.read_text().splitlines() if line.strip()}
    return (max(classes) + 1 if classes else None), None


search_roots = [Path("/kaggle/input")] if Path("/kaggle/input").exists() else []
if REPO_ROOT:
    search_roots += sorted((REPO_ROOT / "ml" / "data").glob("processed*"))
search_roots += EXTRA_SEARCH
candidates = []
for root in search_roots:
    for d, dirs, _ in os.walk(root):
        d = Path(d)
        if is_dataset_root(d):
            nc, names = read_nc_names(d)
            counts = {s: len(list((d / "images" / s).glob("*.jpg"))) for s in SPLITS}
            candidates.append(dict(path=d, nc=nc, names=names, **counts))
            dirs[:] = []          # jangan turun ke dalam dataset yang sudah dikenali

print(pd.DataFrame(candidates).to_string(index=False) if candidates else "tidak ada kandidat")
c3 = [c for c in candidates if c["nc"] == 3]
pick = c3 if len(c3) == 1 else [c for c in c3 if "3cls" in c["path"].name]
if len(pick) != 1:
    raise RuntimeError("Dataset ambigu atau tidak ditemukan. Kandidat nc==3: "
                       + ", ".join(str(c["path"]) for c in c3) + ". Hapus input yang tidak dipakai atau set DATA_ROOT manual.")
DATA_ROOT = pick[0]["path"]
print(f"\n✅ DATA_ROOT = {DATA_ROOT}")

**2b. Isi dataset**: jumlah gambar, format label, distribusi kelas, dan jumlah **video** per split. Peringatan otomatis
bila suatu kelas di test < 30 sampel atau lying ada di < 10 video (jumlah video lebih menentukan daripada jumlah frame).

In [ ]:
def subset(items, n):
    if not n or len(items) <= n:
        return items
    idx = np.linspace(0, len(items) - 1, n).round().astype(int)
    return [items[i] for i in sorted(set(idx))]


SPLIT_ITEMS = {}
for sp in SPLITS:
    items = []
    for img in sorted((DATA_ROOT / "images" / sp).glob("*.jpg")):
        lf = DATA_ROOT / "labels" / sp / f"{img.stem}.txt"
        lines = [line for line in lf.read_text().splitlines() if line.strip()] if lf.exists() else []
        items.append(dict(stem=img.stem, img=img, lines=lines, cls=int(lines[0].split()[0]) if lines else -1,
                          video=video_of(img.stem), source=source_of(img.stem)))
    SPLIT_ITEMS[sp] = subset(items, DEBUG_SUBSET)

rows = []
for sp, items in SPLIT_ITEMS.items():
    ncols = sorted({len(line.split()) for it in items for line in it["lines"]})
    no_label = sum(1 for it in items if not it["lines"])
    for c in (0, 1, 2):
        sel = [it for it in items if it["cls"] == c]
        rows.append(dict(split=sp, kelas=CLASS_NAMES[c], frame=len(sel), video=len({it["video"] for it in sel}),
                         mcf=sum(it["source"] == "MCF" for it in sel), urfall=sum(it["source"] == "UR Fall" for it in sel)))
    print(f"{sp:5}: {len(items)} gambar | kolom label {ncols} | tanpa label: {no_label}")
STATS = pd.DataFrame(rows)
print(STATS.to_string(index=False))
for r in STATS[STATS.split == "test"].itertuples():
    if r.frame < 30:
        print(f"⚠ test/{r.kelas}: hanya {r.frame} frame (< 30)")
n_lying_vid_test = int(STATS[(STATS.split == "test") & (STATS.kelas == "lying_on_ground")].video.iloc[0])
if n_lying_vid_test < 10:
    print(f"⚠ lying di test hanya ada di {n_lying_vid_test} video (< 10): semua metrik lying punya ketidakpastian besar")
if DEBUG_SUBSET:
    print(f"⚠ DEBUG_SUBSET={DEBUG_SUBSET}: hanya sebagian gambar dipakai; angka tidak bermakna")

## 3. data.yaml dan dataset kerja
Input Kaggle bersifat read-only, jadi dataset kerja dibuat di `WORK/ds_pose`: gambar ditautkan **per berkas** (symlink;
salin bila symlink gagal) dan label pose baru ditulis di bagian 4a. Dataset asli tidak diubah. Symlink harus per berkas:
ultralytics me-*resolve* path folder split, sehingga folder yang di-symlink akan membuatnya membaca label 5 kolom asli.

In [ ]:
import shutil

DS_POSE = WORK / "ds_pose"
n_link = n_copy = 0
for sp, items in SPLIT_ITEMS.items():
    (DS_POSE / "images" / sp).mkdir(parents=True, exist_ok=True)
    (DS_POSE / "labels" / sp).mkdir(parents=True, exist_ok=True)
    for it in items:
        dst = DS_POSE / "images" / sp / it["img"].name
        if dst.exists() or dst.is_symlink():
            continue
        try:
            os.symlink(it["img"].resolve(), dst)
            n_link += 1
        except OSError:
            shutil.copy2(it["img"], dst)
            n_copy += 1
print(f"gambar baru ditautkan: {n_link} | disalin: {n_copy}")
for sp in SPLITS:
    assert len(list((DS_POSE / "images" / sp).glob("*.jpg"))) == len(SPLIT_ITEMS[sp]), f"jumlah gambar {sp} tidak cocok"

DATA_YAML = DS_POSE / "data.yaml"
DATA_YAML.write_text(yaml.safe_dump({
    "path": str(DS_POSE), "train": "images/train", "val": "images/val", "test": "images/test",
    "nc": 3, "names": CLASS_NAMES, "kpt_shape": [17, 3], "flip_idx": FLIP_IDX}, sort_keys=False))
print(DATA_YAML.read_text())

**3b. Fungsi verifikasi jumlah gambar yang terbaca ultralytics.** Cache label tidak boleh membuang gambar diam-diam.
Fungsi ini **dipanggil di akhir 4a**, karena label 56 kolom baru ada setelah pseudo-label. Ia juga memastikan label yang
dibaca berasal dari `ds_pose`, bukan dari dataset asli.

In [ ]:
def verify_ultralytics_counts(task="pose"):
    from ultralytics.data.dataset import YOLODataset
    data = {"names": CLASS_NAMES, "nc": 3, "kpt_shape": [17, 3], "flip_idx": FLIP_IDX, "channels": 3}
    for sp in SPLITS:
        n_files = len(list((DS_POSE / "images" / sp).glob("*.jpg")))
        ds = YOLODataset(img_path=str(DS_POSE / "images" / sp), data=data, task=task, imgsz=640, augment=False)
        n_lab = sum(len(lb["cls"]) > 0 for lb in ds.labels)
        assert str(DS_POSE) in ds.label_files[0], f"label dibaca dari luar ds_pose: {ds.label_files[0]}"
        assert len(ds) == n_files, f"{sp}: ultralytics membaca {len(ds)} dari {n_files} gambar"
        assert n_lab == n_files, f"{sp}: hanya {n_lab} gambar punya label terbaca"
        print(f"✅ {task}/{sp}: {len(ds)} gambar terbaca = {n_files} berkas; semua berlabel")

## 4. Arsitektur: YOLO11-Pose
**Istilah singkat.** *CNN* (convolutional neural network) adalah jaringan yang lapisannya berupa filter konvolusi yang
digeser di atas gambar; lapisan awal menangkap tepi dan tekstur, lapisan dalam menangkap bentuk. *YOLO* adalah detektor
satu-lintasan: CNN (backbone → neck) lalu *head* yang, di setiap lokasi grid, memprediksi kotak, skor kelas, dan
(untuk varian pose) 17 keypoint × (x, y, visibilitas). Bandingkan dengan MLP: head YOLO pada dasarnya lapisan konvolusi
1×1, yaitu MLP kecil yang dipakai bersama di setiap lokasi.

**Alasan kelompok:** skeleton memberi sinyal postur tambahan. **Status: hipotesis yang diuji, bukan hasil.**
Hal yang membatasi hipotesis ini:
- Kelas tetap diputuskan oleh kepala kelas yang sama dengan model detect. Keypoint hanya menambah loss saat training
  (multi-task); ia tidak menyaring kelas saat inferensi.
- Keypoint latih berasal dari teacher, bukan anotasi manusia. Teacher sekeluarga dengan pembuat box MCF (yolo11m@960),
  jadi match di MCF akan tampak tinggi.
- Ultralytics menghitung loss *posisi* keypoint hanya untuk keypoint terlihat (v > 0), tetapi loss *visibilitas* memakai
  semua keypoint. Frame tanpa match (semua v=0) ikut mengajari model bahwa "tidak ada sendi terlihat" pada frame itu.
- Ultralytics memilih `best.pt` dengan *fitness* yang pada model pose ikut menghitung mAP pose di val. Keypoint val
  berasal dari teacher, jadi pemilihan checkpoint sebagian menghadiahi kesesuaian dengan teacher. Pemilihan konfigurasi
  di notebook ini tetap memakai AP lying-vs-normal di val, bukan fitness ultralytics.

### 4a. Pseudo-label keypoint dari teacher
Untuk setiap gambar: teacher mendeteksi orang (imgsz 960, conf 0,25), lalu dipilih orang dengan IoU tertinggi terhadap
box label asli. Match sah bila IoU ≥ 0,5 dan conf orang ≥ 0,3. Baris label = kelas + box **asli** + 17 keypoint teacher.

**Gambar tanpa match tidak dibuang**; tetap ditulis 56 kolom dengan semua keypoint v=0. Alasannya:
(i) membuang gambar yang gagal dibaca teacher berarti membuang justru frame lying tersulit, dan set train/test menjadi
berbeda dari baseline sehingga hasil tidak sebanding; (ii) loss box/kelas tetap berjalan untuk gambar itu.
Hasil disimpan di `WORK/ds_pose/labels` dan `WORK/pseudo_meta.csv`; sel ini dilewati bila hasilnya sudah ada.

In [ ]:
META = WORK / "pseudo_meta.csv"


def pseudo_done():
    if not META.exists():
        return False
    return all(len(list((DS_POSE / "labels" / sp).glob("*.txt"))) == len(SPLIT_ITEMS[sp]) for sp in SPLITS)


if RUN_PSEUDO_LABEL and (FORCE_RECOMPUTE or not pseudo_done()):
    teacher = YOLO(TEACHER)
    rows = []
    for sp in SPLITS:
        items, out_dir = SPLIT_ITEMS[sp], DS_POSE / "labels" / sp
        for i in range(0, len(items), 16):
            chunk = items[i:i + 16]
            results = teacher.predict([str(it["img"]) for it in chunk], imgsz=TEACHER_IMGSZ, conf=TEACHER_CONF,
                                      classes=[0], device=DEVICE, verbose=False)
            for it, r in zip(chunk, results, strict=True):
                has = r.boxes is not None and len(r.boxes) > 0
                boxes = r.boxes.xyxyn.cpu().numpy() if has else np.zeros((0, 4))
                pconf = r.boxes.conf.cpu().numpy() if has else np.zeros(0)
                kxy = r.keypoints.xyn.cpu().numpy() if has and r.keypoints is not None else None
                kcf = r.keypoints.conf.cpu().numpy() if has and r.keypoints is not None and r.keypoints.conf is not None else None
                out_lines = []
                for line in it["lines"]:
                    gt_box = xywhn_to_xyxy([float(x) for x in line.split()[1:5]])
                    ious = [iou_xyxy(gt_box, b) for b in boxes]
                    j = int(np.argmax(ious)) if ious else -1
                    iou = ious[j] if j >= 0 else 0.0
                    ok = j >= 0 and iou >= MATCH_IOU and pconf[j] >= MATCH_CONF and kxy is not None and kcf is not None
                    row = build_pose_row(line, kxy[j], kcf[j], KP_V2, KP_V1) if ok else build_pose_row(line)
                    out_lines.append(row)
                    rows.append(dict(stem=it["stem"], split=sp, cls=int(line.split()[0]), source=it["source"],
                                     video=it["video"], matched=bool(ok), iou=round(float(iou), 4),
                                     person_conf=round(float(pconf[j]), 4) if j >= 0 else 0.0,
                                     n_vis=n_visible(row), mean_kconf=round(float(kcf[j].mean()), 4) if ok else np.nan))
                (out_dir / f"{it['stem']}.txt").write_text("\n".join(out_lines) + "\n")
            if (i // 16) % 50 == 0:
                print(f"  {sp}: {min(i + 16, len(items))}/{len(items)}")
    pd.DataFrame(rows).to_csv(META, index=False)
    for c in (DS_POSE / "labels").glob("*.cache"):
        c.unlink()                       # cache lama akan menyembunyikan label baru
    print("pseudo-label selesai")
elif pseudo_done():
    print(f"pseudo-label sudah ada ({META}); dilewati. FORCE_RECOMPUTE=True untuk menghitung ulang.")
else:
    print("RUN_PSEUDO_LABEL=False dan label pose belum ada: bagian 4b-6 tidak bisa dijalankan.")
PSEUDO = pd.read_csv(META) if META.exists() else None

**4a-2. Laporan kualitas pseudo-label** per split, sumber, dan kelas: persentase match, rata-rata conf keypoint (frame
yang match), dan persentase frame dengan < 8 keypoint terlihat (v ≥ 1). MCF dan UR Fall dipisah karena teacher
sekeluarga dengan pembuat box MCF.

In [ ]:
if PSEUDO is not None:
    rep = (PSEUDO.assign(lt8=PSEUDO.n_vis < 8, kelas=PSEUDO.cls.map(CLASS_NAMES))
           .groupby(["split", "source", "kelas"], sort=False)
           .agg(frame=("stem", "size"), match_pct=("matched", "mean"), conf_kp=("mean_kconf", "mean"),
                kp_kurang8_pct=("lt8", "mean")))
    rep["match_pct"] = (100 * rep.match_pct).round(1)
    rep["kp_kurang8_pct"] = (100 * rep.kp_kurang8_pct).round(1)
    print(rep.round(3).to_string())
    lt = PSEUDO[(PSEUDO.split == "train") & (PSEUDO.cls == GT_LYING)]
    rate = lt.matched.mean() if len(lt) else float("nan")
    print(f"\nmatch lying di train: {rate:.1%}")
    if rate < 0.60:
        print("⚠ match lying train < 60%: keypoint hampir hanya dipelajari dari frame non-lying")
    print("Catatan: teacher sekeluarga dengan pembuat box MCF (yolo11m@960); match MCF yang tinggi tidak membuktikan keypoint benar.")

**4a-3. Verifikasi jumlah gambar yang terbaca ultralytics** (fungsi dari 3b), untuk task pose dan detect.

In [ ]:
if pseudo_done():
    verify_ultralytics_counts("pose")
    verify_ultralytics_counts("detect")
else:
    print("label pose belum lengkap; verifikasi dilewati")

**4a-4. Audit visual wajib.** 24 frame lying acak dari train (12 MCF, 12 UR Fall) dengan skeleton dari berkas label
yang **akan dipelajari model**. Disimpan ke `WORK/diag/pseudo_kp_lying.png`. Status: **belum dinilai** sampai Laila
menilainya (keypoint masuk akal? sendi tertukar? ada skeleton milik orang lain?).

In [ ]:
import cv2
import matplotlib.pyplot as plt

CLASS_COLOR = {0: "#2a78d6", 1: "#eb6834", 2: "#1baf7a"}


def draw_pose(ax, img_path, label_path, title):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    ax.imshow(img)
    ax.axis("off")
    for line in label_path.read_text().splitlines():
        p = [float(x) for x in line.split()]
        x0, y0, x1, y1 = xywhn_to_xyxy(p[1:5])
        ax.add_patch(plt.Rectangle((x0 * w, y0 * h), (x1 - x0) * w, (y1 - y0) * h, fill=False,
                                   lw=1.5, ec=CLASS_COLOR[int(p[0])]))
        k = np.array(p[5:]).reshape(17, 3)
        for a, b in COCO_SKELETON:
            if k[a, 2] > 0 and k[b, 2] > 0:
                ax.plot([k[a, 0] * w, k[b, 0] * w], [k[a, 1] * h, k[b, 1] * h], "-", lw=1.2, c="#e34948")
        vis = k[:, 2] > 0
        ax.scatter(k[vis, 0] * w, k[vis, 1] * h, s=8, c=np.where(k[vis, 2] == 2, "#ffd400", "#ffffff"))
    ax.set_title(title, fontsize=7, loc="left")


if PSEUDO is not None:
    rng = np.random.default_rng(SEED)
    pool = PSEUDO[(PSEUDO.split == "train") & (PSEUDO.cls == GT_LYING)]
    pick = []
    for src in ("MCF", "UR Fall"):
        cand = pool[pool.source == src]
        pick += cand.iloc[rng.permutation(len(cand))[:12]].to_dict("records")
    if pick:
        cols = 6
        fig, axes = plt.subplots(int(np.ceil(len(pick) / cols)), cols, figsize=(18, 3.2 * np.ceil(len(pick) / cols)))
        for ax in np.atleast_1d(axes).ravel():
            ax.axis("off")
        for ax, r in zip(np.atleast_1d(axes).ravel(), pick, strict=False):   # sumbu grid bisa lebih banyak dari gambar
            draw_pose(ax, DS_POSE / "images" / "train" / f"{r['stem']}.jpg", DS_POSE / "labels" / "train" / f"{r['stem']}.txt",
                      f"{r['stem']}\nmatch={r['matched']} IoU={r['iou']:.2f} kp={r['n_vis']}")
        fig.suptitle("Pseudo-keypoint pada frame lying (kuning v=2, putih v=1) — STATUS: BELUM DINILAI", fontsize=11)
        plt.tight_layout()
        fig.savefig(DIAG / "pseudo_kp_lying.png", dpi=110)
        plt.show()
        print(f"Tersimpan: {DIAG / 'pseudo_kp_lying.png'} | Status penilaian: BELUM DINILAI (menunggu Laila)")

### 4b. Eksplorasi ringan
**Pemetaan teori (MLP/RNN/LSTM) ke parameter nyata YOLO**

| Konsep teori | Padanan di sini | Catatan |
|---|---|---|
| Jumlah hidden layer & hidden unit | varian `yolo11n-pose` vs `yolo11s-pose` (kedalaman/lebar) dan jumlah layer beku (`freeze=0/10`) | `freeze` membekukan layer awal: bobot pretrained dipakai apa adanya |
| Fungsi aktivasi | SiLU (`x·sigmoid(x)`) di seluruh YOLO11 | tidak diganti; perannya sama seperti di MLP: memberi non-linearitas |
| Epoch | `epochs` + `patience` (berhenti bila val tidak membaik) | anggaran eksplorasi 30 epoch, patience 10 |
| Sequence length (RNN/LSTM) | **tidak ada padanan**: model ini satu frame | hanya smoothing N sampel saat evaluasi (lihat `temporal_velocity.ipynb`); jarak frame tidak seragam dan fps MCF belum terverifikasi, jadi satuannya "N sampel", bukan detik |
| Learning rate, optimizer | `lr0`, `optimizer` (AdamW vs SGD) | SGD diuji dengan lr0 bawaannya (0,01) sebagai satu perubahan |
| (khas CNN) resolusi input | `imgsz` 640 vs 800 | biaya komputasi naik ≈ kuadrat |

**Desain run** (maks 6, seed 0, satu perubahan per run dari R1). R1 memakai resep baseline dengan kepala pose.
**Penyimpangan yang disengaja:** baseline dilatih 70 epoch, sedangkan anggaran eksplorasi 30 epoch. Karena itu ada **R0**,
yaitu baseline detect dengan anggaran yang sama. Efek kepala pose dibaca dari R1 vs R0, bukan dari R1 vs angka baseline
70 epoch. Varian `lr0` dengan AdamW tidak diuji (anggaran habis).

**Kriteria pemilihan (ditetapkan sebelum hasil):** (1) AP lying-vs-normal di val; (2) recall lying @FPR 5% di val;
(3) bila selisih di bawah noise (selisih < 0,03 atau CI 95% tumpang tindih), pilih model yang lebih kecil/sederhana.
Recall bawaan ultralytics (titik kerja F1-max) tidak dipakai karena tidak mengontrol alarm palsu. Hasil otomatis hanya
**kandidat**; keputusan di tangan Laila.

In [ ]:
# Perkiraan waktu (KASAR, bukan pengukuran): acuan tim yolo11n detect 3 kelas, 41 epoch ~ 29 menit di T4.
MIN_PER_EPOCH_REF = 29 / 41
MODEL_FACTOR = {"yolo11n.pt": 1.0, "yolo11n-pose.pt": 1.1, "yolo11s-pose.pt": 1.9}   # asumsi
frac = len(SPLIT_ITEMS["train"]) / 3768


def est_minutes(cfg, epochs):
    f = MODEL_FACTOR.get(cfg["model"], 1.5) * (cfg.get("imgsz", 640) / 640) ** 2 * (0.85 if cfg.get("freeze") else 1.0)
    return MIN_PER_EPOCH_REF * f * epochs * frac


tot = 0
for cfg in EXPLORE_RUNS:
    m = est_minutes(cfg, EXPLORE_EPOCHS)
    tot += m
    print(f"{cfg['rid']}: {cfg['model']:<16} {cfg['note']:<55} ≈ {m:5.0f} menit (batas atas)")
print(f"Total eksplorasi ≈ {tot / 60:.1f} jam (patience dapat mempercepat). Kaggle: sesi maks 12 jam, kuota GPU mingguan terbatas.")

**4b-2. Fungsi bantu training dan skor.** `train_run` melewati run yang sudah selesai (berguna bila sesi Kaggle terputus)
dan menurunkan `batch` lalu `imgsz` bila kehabisan memori GPU (OOM); perubahan itu dicatat karena membuat run tidak lagi
identik. Skor lying per gambar memakai `eval_lying.predict_scores` tanpa perubahan.

In [ ]:
import json
import time

RUNS = WORK / "runs"


def run_name(rid, suffix=""):
    return f"{TAG}_{RUN_ID_START:03d}_{rid}{suffix}"


def hp_for(cfg, epochs, patience, **over):
    hp = {**BASE_HP, **AUG, "epochs": epochs, "patience": patience}
    hp.update({k: cfg[k] for k in ("freeze", "imgsz", "optimizer", "lr0", "batch") if k in cfg})
    hp.update(over)
    return hp


def train_run(name, weights, hp, allow_train=True):
    d = RUNS / name
    best = d / "weights" / "best.pt"
    if best.exists() and (d / "results.csv").exists() and not FORCE_RECOMPUTE:
        eff = yaml.safe_load((d / "args.yaml").read_text())
        print(f"[lewati] {name} sudah selesai")
        return d, {k: eff.get(k) for k in ("batch", "imgsz", "epochs", "freeze", "lr0", "optimizer", "seed")}
    if not allow_train or not (HAS_GPU or ALLOW_CPU_TRAIN):
        print(f"[belum dijalankan] {name}: " + ("saklar mati" if not allow_train else "tanpa GPU"))
        return None, None
    hp = dict(hp)
    while True:
        try:
            t0 = time.time()
            model = YOLO(str(weights))
            model.train(data=str(DATA_YAML), project=str(RUNS), name=name, exist_ok=True, device=DEVICE,
                        verbose=False, **hp)
            print(f"[selesai] {name} dalam {(time.time() - t0) / 60:.1f} menit")
            (Path(model.trainer.save_dir) / "hp_efektif.json").write_text(json.dumps(hp, indent=2))
            return Path(model.trainer.save_dir), hp
        except RuntimeError as e:
            if "out of memory" not in str(e).lower():
                raise
            torch.cuda.empty_cache()
            if hp["batch"] > 4:
                hp["batch"] //= 2
            elif hp["imgsz"] > 640:
                hp["imgsz"] = 640
            else:
                raise
            print(f"[OOM] {name}: ulang dengan batch={hp['batch']} imgsz={hp['imgsz']} (run TIDAK lagi identik)")


def score_split(weights, sp, imgsz, out_csv):
    out_csv = Path(out_csv)
    if out_csv.exists() and not FORCE_RECOMPUTE:
        return pd.read_csv(out_csv)
    items = SPLIT_ITEMS[sp]
    s = eval_lying.predict_scores(str(weights), [it["img"] for it in items], imgsz, DEVICE, 32)
    df = pd.DataFrame(dict(stem=[it["stem"] for it in items], gt=[it["cls"] for it in items],
                           video=[it["video"] for it in items], score=np.round(s, 5)))
    df.to_csv(out_csv, index=False)
    return df


def lying_summary(df):
    s, g, v = df["score"].to_numpy(), df["gt"].to_numpy(), df["video"].to_numpy()
    ap, ap_lo, ap_hi = bootstrap_by_video(ap_fn, s, g, v, N_BOOT, SEED)
    r5, r5_lo, r5_hi = bootstrap_by_video(make_recall_fpr_fn(0.05), s, g, v, N_BOOT, SEED)
    return dict(ap=ap, ap_lo=ap_lo, ap_hi=ap_hi, rec5=r5, rec5_lo=r5_lo, rec5_hi=r5_hi,
                n_vid_lying=int(df.loc[df["gt"] == GT_LYING, "video"].nunique()))


def box_metrics(weights, sp, imgsz, cache_json):
    cache_json = Path(cache_json)
    if cache_json.exists() and not FORCE_RECOMPUTE:
        return json.loads(cache_json.read_text())
    m = YOLO(str(weights)).val(data=str(DATA_YAML), split=sp, imgsz=imgsz, batch=16, device=DEVICE,
                               plots=False, verbose=False)
    out = dict(map50=float(m.box.map50), map=float(m.box.map), speed=dict(m.speed))
    cache_json.write_text(json.dumps(out))
    return out


def model_size(weights):
    # YOLO.info(verbose=False) mengembalikan None di ultralytics 8.4, jadi pakai fungsi utilitasnya langsung
    from ultralytics.utils.torch_utils import get_flops, get_num_params
    m = YOLO(str(weights)).model
    return dict(params_M=get_num_params(m) / 1e6, gflops=float(get_flops(m)), size_MB=Path(weights).stat().st_size / 1e6)

**4b-3. Menjalankan eksplorasi dan mengukur di val.** Tabel: konfigurasi vs metrik val (AP dan recall@FPR 5% dengan
CI bootstrap per video, mAP50 box val). R1 vs R0 diuji berpasangan untuk melihat efek kepala pose pada anggaran sama.

In [ ]:
EXP_ROWS = {}
for cfg in EXPLORE_RUNS:
    name = run_name(cfg["rid"])
    d, hp_eff = train_run(name, cfg["model"], hp_for(cfg, EXPLORE_EPOCHS, EXPLORE_PATIENCE), allow_train=RUN_EXPLORATION)
    row = dict(rid=cfg["rid"], model=cfg["model"], perubahan=cfg["note"], status="belum dijalankan")
    if d is not None:
        best = d / "weights" / "best.pt"
        df = score_split(best, "val", hp_eff["imgsz"], d / "scores_val.csv")
        row.update(status="selesai", imgsz=hp_eff["imgsz"], batch=hp_eff["batch"], freeze=hp_eff.get("freeze") or 0,
                   optimizer=hp_eff.get("optimizer"), lr0=hp_eff.get("lr0"), weights=str(best),
                   **lying_summary(df), map50_val=box_metrics(best, "val", hp_eff["imgsz"], d / "box_val.json")["map50"],
                   **model_size(best))
    EXP_ROWS[cfg["rid"]] = row
EXP = pd.DataFrame(EXP_ROWS.values())
show_cols = [c for c in ["rid", "model", "perubahan", "status", "imgsz", "batch", "ap", "ap_lo", "ap_hi", "rec5",
                         "rec5_lo", "rec5_hi", "map50_val", "params_M", "n_vid_lying"] if c in EXP.columns]
print(EXP[show_cols].round(3).to_string(index=False))

if {"R0", "R1"} <= set(EXP.loc[EXP.status == "selesai", "rid"]):
    v0 = pd.read_csv(RUNS / run_name("R0") / "scores_val.csv")
    v1 = pd.read_csv(RUNS / run_name("R1") / "scores_val.csv")
    assert (v0["stem"].to_numpy() == v1["stem"].to_numpy()).all()
    pb = paired_bootstrap(ap_fn, v0["score"].to_numpy(), v1["score"].to_numpy(), v0["gt"].to_numpy(),
                          v0["video"].to_numpy(), N_BOOT, SEED)
    print(f"\nEfek kepala pose (R1 - R0, val, anggaran sama): ΔAP = {pb['delta']:+.3f} "
          f"[{pb['lo']:+.3f}, {pb['hi']:+.3f}], P(R1 > R0) = {pb['p']:.2f}")
    show_interp("paired", pb)   # CI bootstrap per video sudah memuat ketidakpastian jumlah video

**4b-4. Kandidat dari eksplorasi** (hanya run pose; R0 adalah kontrol). Kriteria dicetak bersama hasilnya.
Bila tidak ada run yang selesai, kandidat = `None` dan bagian berikutnya melewati training.

In [ ]:
def overlap(a_lo, a_hi, b_lo, b_hi):
    return not (a_hi < b_lo or a_lo > b_hi)


CANDIDATE = None
pool = EXP[(EXP.status == "selesai") & (EXP.rid != "R0")] if "ap" in EXP.columns else EXP.iloc[0:0]
if pool.empty:
    print("Belum ada run pose yang selesai: kandidat belum bisa dipilih (belum diuji).")
else:
    print("Kriteria: (1) AP val; (2) recall@FPR5% val; (3) selisih < 0,03 atau CI tumpang tindih -> model lebih kecil/sederhana.")
    top = pool.sort_values("ap", ascending=False).iloc[0]
    near = pool[pool.apply(lambda r: abs(r.ap - top.ap) < 0.03 or overlap(r.ap_lo, r.ap_hi, top.ap_lo, top.ap_hi), axis=1)]
    top_r = near.sort_values("rec5", ascending=False).iloc[0]
    near = near[near.apply(lambda r: abs(r.rec5 - top_r.rec5) < 0.03 or overlap(r.rec5_lo, r.rec5_hi, top_r.rec5_lo, top_r.rec5_hi), axis=1)]
    simplest = near.assign(_base=(near.rid != "R1").astype(int)).sort_values(["params_M", "imgsz", "freeze", "_base"]).iloc[0]
    CANDIDATE = next(c for c in EXPLORE_RUNS if c["rid"] == simplest.rid)
    print(f"AP tertinggi: {top.rid} ({top.ap:.3f}); dalam noise: {list(near.rid)}")
    print(f"➡ KANDIDAT (otomatis, menunggu keputusan Laila): {CANDIDATE['rid']} — {CANDIDATE['note']}")

## 5. Fine-tuning: one-stage penuh vs two-stage
**Yang dipelajari dari riwayat `EXPERIMENTS.md`:**
- EXP-007 (two-stage: freeze 10 selama 25 epoch, lalu unfreeze dengan lr0 0,0001) kalah dari EXP-006 (one-stage).
  Namun keduanya diukur pada **dataset lama** dengan lying hanya **1 gambar** di test, jadi tidak sebanding dengan v2 dan
  tidak membuktikan apa pun tentang two-stage di v2.
- Di v2, freeze=10 satu tahap memberi mAP50 test 0,592 vs 0,58 (selisih dalam noise; AP lying-vs-normal belum dihitung).
- EXP-002/004/008 (model lebih besar) memburuk di dataset lama 924 gambar; dataset v2 lebih besar (3.768 train),
  jadi kesimpulan "nano paling optimal" belum tentu berlaku. Itu sebabnya R2 (`yolo11s-pose`) diuji, bukan diasumsikan.

**Keputusan:** two-stage **tidak dianggap lebih baik** sebelum diukur. Bagian ini melatih kandidat dengan jadwal penuh
dalam dua cara dengan total epoch sama (70): (a) one-stage 70 epoch, patience 30; (b) Stage 1 freeze 10 selama 20 epoch,
lalu Stage 2 unfreeze 50 epoch dengan lr0 × 0,1 (pola template). Dipilih two-stage hanya bila **lebih baik secara
berpasangan** di val (P ≥ 0,9 dan CI tidak memuat 0); seri → one-stage (lebih sederhana).

**Augmentasi:** tetap bawaan ultralytics (sama dengan baseline v2) agar selisih bisa dikaitkan ke arsitektur:
hsv, scale 0,5, translate 0,1, mosaic. `fliplr=0,5` aman karena `flip_idx` menukar keypoint kiri-kanan dengan benar;
`flipud=0` dan `degrees=0` (orang berdiri tidak boleh berubah menjadi berbaring). Augmentasi EXP-006 (hsv_v 0,6, bgr 0,2,
erasing 0,4) tidak diambil: efeknya tidak pernah diisolasi, dan diukur di dataset lama.

In [ ]:
FT = {}
if CANDIDATE is None:
    print("Kandidat belum ada: bagian 5 belum dijalankan.")
else:
    lr_c = CANDIDATE.get("lr0", BASE_HP["lr0"])
    # (a) one-stage penuh
    d, hp = train_run(run_name(CANDIDATE["rid"], "_full"), CANDIDATE["model"],
                      hp_for(CANDIDATE, FINAL_EPOCHS, FINAL_PATIENCE), allow_train=RUN_FINETUNE)
    if d is not None:
        FT["one-stage"] = dict(dir=d, weights=d / "weights" / "best.pt", imgsz=hp["imgsz"])
    # (b) two-stage
    d1, hp1 = train_run(run_name(CANDIDATE["rid"], "_ts1"), CANDIDATE["model"],
                        hp_for(CANDIDATE, STAGE1_EPOCHS, STAGE1_EPOCHS, freeze=STAGE1_FREEZE), allow_train=RUN_FINETUNE)
    if d1 is not None:
        d2, hp2 = train_run(run_name(CANDIDATE["rid"], "_ts2"), d1 / "weights" / "best.pt",
                            hp_for(CANDIDATE, STAGE2_EPOCHS, STAGE2_PATIENCE, freeze=0, lr0=lr_c * STAGE2_LR_FACTOR),
                            allow_train=RUN_FINETUNE)
        if d2 is not None:
            FT["two-stage"] = dict(dir=d2, weights=d2 / "weights" / "best.pt", imgsz=hp2["imgsz"])
    for k, v in FT.items():
        v["val"] = score_split(v["weights"], "val", v["imgsz"], v["dir"] / "scores_val.csv")
        v["summary"] = lying_summary(v["val"])
        print(f"{k:10}: AP val {v['summary']['ap']:.3f} [{v['summary']['ap_lo']:.3f}, {v['summary']['ap_hi']:.3f}] | "
              f"recall@FPR5% {v['summary']['rec5']:.3f}")

**5b. Memilih skema** (aturan di atas), lalu mengulang **3 seed** hanya untuk skema terpilih. Model untuk test ditetapkan
**sebelum** melihat test: seed 0. Seed 1 dan 2 hanya untuk melihat kestabilan (rata-rata ± simpangan baku di val).

In [ ]:
FINAL_WEIGHTS = FINAL_IMGSZ = FINAL_SCHEME = None
if "one-stage" in FT:
    FINAL_SCHEME = "one-stage"
    if "two-stage" in FT:
        a, b = FT["one-stage"]["val"], FT["two-stage"]["val"]
        assert (a["stem"].to_numpy() == b["stem"].to_numpy()).all()
        pb = paired_bootstrap(ap_fn, a["score"].to_numpy(), b["score"].to_numpy(), a["gt"].to_numpy(),
                              a["video"].to_numpy(), N_BOOT, SEED)
        lab = show_interp("paired", pb)
        print(f"two-stage - one-stage (val): ΔAP {pb['delta']:+.3f} [{pb['lo']:+.3f}, {pb['hi']:+.3f}], P={pb['p']:.2f}")
        if lab == "LEBIH BAIK":
            FINAL_SCHEME = "two-stage"
    FINAL_WEIGHTS, FINAL_IMGSZ = FT[FINAL_SCHEME]["weights"], FT[FINAL_SCHEME]["imgsz"]
    print(f"➡ skema terpilih: {FINAL_SCHEME} | bobot untuk test (seed 0): {FINAL_WEIGHTS}")

SEED_ROWS = []
if FINAL_SCHEME and RUN_SEEDS:
    for sd in SEEDS:
        if sd == SEED:
            s = FT[FINAL_SCHEME]["summary"]
        else:
            if FINAL_SCHEME == "one-stage":
                d, hp = train_run(run_name(CANDIDATE["rid"], f"_full_s{sd}"), CANDIDATE["model"],
                                  hp_for(CANDIDATE, FINAL_EPOCHS, FINAL_PATIENCE, seed=sd))
            else:
                d1, _ = train_run(run_name(CANDIDATE["rid"], f"_ts1_s{sd}"), CANDIDATE["model"],
                                  hp_for(CANDIDATE, STAGE1_EPOCHS, STAGE1_EPOCHS, freeze=STAGE1_FREEZE, seed=sd))
                d, hp = (None, None) if d1 is None else train_run(
                    run_name(CANDIDATE["rid"], f"_ts2_s{sd}"), d1 / "weights" / "best.pt",
                    hp_for(CANDIDATE, STAGE2_EPOCHS, STAGE2_PATIENCE, freeze=0,
                           lr0=CANDIDATE.get("lr0", BASE_HP["lr0"]) * STAGE2_LR_FACTOR, seed=sd))
            if d is None:
                continue
            s = lying_summary(score_split(d / "weights" / "best.pt", "val", hp["imgsz"], d / "scores_val.csv"))
        SEED_ROWS.append(dict(seed=sd, ap_val=s["ap"], rec5_val=s["rec5"]))
    if SEED_ROWS:
        sdf = pd.DataFrame(SEED_ROWS)
        print(sdf.round(3).to_string(index=False))
        print(f"rata-rata ± sd: AP {sdf.ap_val.mean():.3f} ± {sdf.ap_val.std(ddof=1) if len(sdf) > 1 else float('nan'):.3f} | "
              f"recall@FPR5% {sdf.rec5_val.mean():.3f} ± {sdf.rec5_val.std(ddof=1) if len(sdf) > 1 else float('nan'):.3f}")

if FINAL_WEIGHTS_OVERRIDE:          # diatur di sel Pengaturan
    FINAL_WEIGHTS, FINAL_IMGSZ, FINAL_SCHEME = Path(FINAL_WEIGHTS_OVERRIDE), FINAL_IMGSZ_OVERRIDE, "override"
READY = FINAL_WEIGHTS is not None and Path(FINAL_WEIGHTS).exists()
print("Siap evaluasi test:", READY)

## 6. Evaluasi test set
Aturan yang dipegang di seluruh bagian ini:
- **Ambang tidak dipilih dari test.** Titik kerja = ambang yang memberi FPR frame-normal ≤ 1% / ≤ 5% di **val**.
- Recall @FPR x% di test (dengan ambang dihitung dari test) hanya **deskriptif** dan dihitung sama untuk baseline dan
  kandidat, karena skor val baseline tidak tersedia. Ia dipakai untuk membandingkan, bukan untuk memilih ambang.
- Perbandingan hanya terhadap **baseline v2** (`eval_3cls.scores.csv`), berpasangan per gambar dan di-bootstrap per video.
  Angka eksperimen lama hanya ditampilkan di kolom berlabel "dataset lama".
- **Aturan terima** (ditetapkan sebelum hasil): ΔAP berpasangan > 0 dengan P(kandidat > baseline) ≥ 0,9; recall@FPR 10%
  tidak turun; tidak ada skenario runtuh; alarm palsu s23 turun. Seri → model yang lebih sederhana/kecil.

**6a. Skor lying per gambar** (val dan test) dengan `eval_lying.predict_scores`, lalu dipasangkan dengan baseline per nama berkas.

In [ ]:
EVAL = WORK / "eval"
EVAL.mkdir(exist_ok=True)
RESULT = {}
if not READY:
    print("Belum dijalankan: bobot final belum ada (butuh GPU di Kaggle, atau isi FINAL_WEIGHTS_OVERRIDE).")
else:
    VAL = score_split(FINAL_WEIGHTS, "val", FINAL_IMGSZ, EVAL / "cand_scores_val.csv")
    TEST = score_split(FINAL_WEIGHTS, "test", FINAL_IMGSZ, EVAL / "cand_scores_test.csv")
    PAIR = TEST.merge(BASE[["stem", "gt", "score"]].rename(columns={"gt": "gt_base", "score": "score_base"}),
                      on="stem", how="inner")
    if DEBUG_SUBSET:
        print(f"⚠ DEBUG_SUBSET: hanya {len(PAIR)} gambar test yang dipasangkan")
    else:
        assert len(PAIR) == len(TEST) == len(BASE), "gambar test kandidat dan baseline tidak sama"
    assert (PAIR["gt"] == PAIR["gt_base"]).all(), "label GT berbeda dengan file baseline"
    S, G, V, SB = (PAIR[c].to_numpy() for c in ("score", "gt", "video", "score_base"))
    N_VID_LYING = int(PAIR.loc[PAIR["gt"] == GT_LYING, "video"].nunique())
    ev = eval_lying.evaluate(S, G, V, n_boot=N_BOOT)
    assert abs(ev["ap"] - ap_lying(S[G == GT_LYING], S[G == GT_NORMAL])) < 1e-12
    eval_lying.print_report(ev, f"kandidat ({FINAL_SCHEME}) — laporan eval_lying, ambang conf tetap")

**6b. Titik kerja dari val** (FPR frame-normal ≤ 1% dan ≤ 5% di val), diterapkan ke test.

In [ ]:
if READY:
    sv, gv = VAL["score"].to_numpy(), VAL["gt"].to_numpy()
    THR = {f: thr_for_fpr(sv[gv == GT_NORMAL], f) for f in TARGET_FPRS}
    rows = []
    for f, t in THR.items():
        rv, rt = rates_at(sv, gv, t), rates_at(S, G, t)
        r, lo, hi = bootstrap_by_video(lambda s, g, t=t: rates_at(s, g, t)["recall"], S, G, V, N_BOOT, SEED)
        rows.append(dict(target_fpr_val=f, ambang=round(t, 4), fpr_val=rv["fpr"], recall_test=rt["recall"],
                         recall_ci=f"[{lo:.2f}, {hi:.2f}]", precision_test=rt["precision"], fp_normal_test=rt["fp_normal"],
                         fpr_test=rt["fpr"], trans_jadi_lying=rt["trans_as_lying"]))
    OP = pd.DataFrame(rows)
    print(OP.round(3).to_string(index=False))
    RESULT["operating_points"] = OP.to_dict("records")

**6c. Metrik standar** (box): mAP50, mAP50-95, Precision, Recall, F1 per kelas dari `model.val(split="test")`;
*confusion matrix* box (mentah dan ternormalisasi); lalu analisis tingkat gambar: kelas kotak dengan confidence
tertinggi vs GT, dan distribusi confidence prediksi benar vs salah. Metrik pose (OKS) bila tercetak hanya
"sesuai teacher", bukan kebenaran.

In [ ]:
if READY:
    MT = YOLO(str(FINAL_WEIGHTS)).val(data=str(DATA_YAML), split="test", imgsz=FINAL_IMGSZ, batch=16, device=DEVICE,
                                      plots=True, project=str(EVAL), name="val_test", exist_ok=True, verbose=False)
    b = MT.box
    per_cls = pd.DataFrame(dict(kelas=[CLASS_NAMES[int(i)] for i in b.ap_class_index], precision=b.p, recall=b.r,
                                f1=2 * b.p * b.r / np.maximum(b.p + b.r, 1e-9), ap50=b.ap50, ap50_95=b.ap))
    print(f"test box: mAP50 {b.map50:.3f} | mAP50-95 {b.map:.3f} | P {b.mp:.3f} | R {b.mr:.3f}")
    print(per_cls.round(3).to_string(index=False))
    if hasattr(MT, "pose"):
        print(f"(pose mAP50 {MT.pose.map50:.3f} — SESUAI TEACHER, bukan kebenaran; tidak dipakai menilai)")
    RESULT["box_test"] = dict(map50=float(b.map50), map50_95=float(b.map), per_class=per_cls.to_dict("records"))

    cm = MT.confusion_matrix.matrix            # [prediksi, GT], indeks terakhir = background
    names = [CLASS_NAMES[i] for i in range(3)] + ["background"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
    for ax, norm in zip(axes, (False, True), strict=True):
        m = cm / np.maximum(cm.sum(0, keepdims=True), 1) if norm else cm
        ax.imshow(m, cmap="Blues")
        for i in range(4):
            for j in range(4):
                ax.text(j, i, f"{m[i, j]:.2f}" if norm else f"{int(m[i, j])}", ha="center", va="center", fontsize=8)
        ax.set_xticks(range(4), names, rotation=30)
        ax.set_yticks(range(4), names)
        ax.set_xlabel("GT")
        ax.set_ylabel("Prediksi")
        ax.set_title("Confusion box " + ("(per kolom GT)" if norm else "(mentah)"))
    plt.tight_layout()
    plt.show()

In [ ]:
def predict_top1(weights, items, imgsz, out_csv):
    out_csv = Path(out_csv)
    if out_csv.exists() and not FORCE_RECOMPUTE:
        return pd.read_csv(out_csv)
    model, rows = YOLO(str(weights)), []
    for i in range(0, len(items), 32):
        chunk = items[i:i + 32]
        for it, r in zip(chunk, model.predict([str(x["img"]) for x in chunk], conf=SCORE_CONF, imgsz=imgsz,
                                              device=DEVICE, verbose=False), strict=True):
            if r.boxes is None or len(r.boxes) == 0:
                rows.append(dict(stem=it["stem"], pred=-1, conf=0.0))
            else:
                k = int(r.boxes.conf.argmax())
                rows.append(dict(stem=it["stem"], pred=int(r.boxes.cls[k]), conf=float(r.boxes.conf[k])))
    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    return df


if READY:
    T1 = predict_top1(FINAL_WEIGHTS, SPLIT_ITEMS["test"], FINAL_IMGSZ, EVAL / "cand_top1_test.csv").merge(
        PAIR[["stem", "gt", "video"]], on="stem")
    lab = {-1: "tidak terdeteksi", **CLASS_NAMES}
    ct = pd.crosstab(T1["gt"].map(CLASS_NAMES), T1.pred.map(lab), rownames=["GT"], colnames=["Prediksi top-1"])
    print("Confusion tingkat gambar (mentah):\n", ct.to_string())
    print("\nTernormalisasi per GT:\n", ct.div(ct.sum(1), axis=0).round(3).to_string())
    def n(g, p):
        return int(((T1["gt"] == g) & (T1.pred == p)).sum())
    print(f"\nError yang menentukan alarm: lying->normal {n(2, 0)} | lying->tidak terdeteksi {n(2, -1)} | "
          f"normal->lying {n(0, 2)} | transitional->lying {n(1, 2)}")
    print(f"Error yang tidak memicu alarm: transitional<->normal {n(1, 0) + n(0, 1)}")
    ok = T1.pred == T1["gt"]
    plt.figure(figsize=(7, 3.4))
    plt.hist([T1.conf[ok], T1.conf[~ok & (T1.pred >= 0)]], bins=20, range=(0, 1), label=["benar", "salah"],
             color=["#2a78d6", "#eb6834"])
    plt.xlabel("confidence kotak top-1")
    plt.ylabel("jumlah gambar")
    plt.legend(frameon=False)
    plt.title("Distribusi confidence: prediksi benar vs salah")
    plt.show()

**6d. Lying vs bukan-lying (ukuran utama).** AP dan recall@FPR 1/3/5/10% (CI bootstrap per video) untuk kandidat dan
baseline; tabel per video pada titik kerja val FPR 5%: recall tiap video lying dan alarm palsu tiap video (termasuk s23).

In [ ]:
if READY:
    rows = []
    for nm, sc in (("baseline v2", SB), ("kandidat", S)):
        ap, lo, hi = bootstrap_by_video(ap_fn, sc, G, V, N_BOOT, SEED)
        row = dict(model=nm, AP=f"{ap:.3f} [{lo:.3f}, {hi:.3f}]")
        for f in REPORT_FPRS:
            r, rlo, rhi = bootstrap_by_video(make_recall_fpr_fn(f), sc, G, V, N_BOOT, SEED)
            row[f"recall@FPR{int(f * 100)}%"] = f"{r:.3f} [{rlo:.2f}, {rhi:.2f}]"
        rows.append(row)
    print("Lying vs normal di test (recall@FPR: ambang dari test, deskriptif, sama untuk kedua model):")
    print(pd.DataFrame(rows).to_string(index=False))

    t5 = THR[0.05]
    pv = []
    for vid, grp in PAIR.groupby("video"):
        ly, nm_ = grp[grp["gt"] == GT_LYING], grp[grp["gt"] == GT_NORMAL]
        pv.append(dict(video=vid, sumber=source_of(grp.stem.iloc[0]), lying=len(ly),
                       recall=(ly.score >= t5).mean() if len(ly) else np.nan,
                       recall_baseline_thr_test=(ly.score_base >= thr_for_fpr(SB[G == GT_NORMAL], 0.05)).mean() if len(ly) else np.nan,
                       normal=len(nm_), alarm_palsu=int((nm_.score >= t5).sum())))
    PV = pd.DataFrame(pv).sort_values(["lying", "alarm_palsu"], ascending=False)
    print(f"\nPer video, titik kerja val FPR5% (ambang {t5:.3f}); kolom baseline memakai ambang FPR5% test-nya sendiri:")
    print(PV.round(3).to_string(index=False))
    tot_rec = rates_at(S, G, t5)["recall"]
    COLLAPSE = show_interp("collapse", dict(total=tot_rec, per_video=PV.dropna(subset=["recall"]).set_index("video").recall.to_dict()))
    FA_CONC = show_interp("fa_concentration", PV.set_index("video").alarm_palsu.to_dict())
    for src in ("MCF", "UR Fall"):
        m = np.array([source_of(x) == src for x in PAIR.stem])
        print(f"AP per sumber {src:8}: kandidat {ap_fn(S[m], G[m]):.3f} | baseline {ap_fn(SB[m], G[m]):.3f}")
    RESULT["per_video"] = PV.to_dict("records")

**6e. Generalisasi.** Gap val→test yang relevan (skenario berbeda; baseline 0,67 → 0,59). Gap train→val hanya informasi:
frame train berurutan dan mirip, jadi mAP train yang tinggi itu wajar.

In [ ]:
if READY:
    bv = box_metrics(FINAL_WEIGHTS, "val", FINAL_IMGSZ, EVAL / "box_val.json")
    gap = 100 * (bv["map50"] - MT.box.map50)
    print(f"mAP50 box: val {bv['map50']:.3f} -> test {MT.box.map50:.3f} (gap {gap:+.1f} poin)")
    GAP_LABEL = show_interp("gap_val_test", abs(gap))
    print(f"AP lying-vs-normal: val {ap_fn(VAL['score'].to_numpy(), VAL['gt'].to_numpy()):.3f} -> test {ap_fn(S, G):.3f}")
    if RUN_TRAIN_SPLIT_VAL:
        bt = box_metrics(FINAL_WEIGHTS, "train", FINAL_IMGSZ, EVAL / "box_train.json")
        print(f"(informasi) mAP50 train {bt['map50']:.3f}; tinggi itu wajar karena frame train berurutan")
    RESULT["generalisasi"] = dict(map50_val=bv["map50"], map50_test=float(MT.box.map50), gap_poin=gap)

**6f. Ketahanan (robustness).** Salinan test yang diganggu (dibuat di memori, tidak disimpan): kecerahan ±30%, blur
Gaussian, noise, JPEG q30, skala 0,75×/1,25×, rotasi ±10°, penutup persegi acak 20% (oklusi parsial); tanpa flip vertikal.
Ambang dari val (FPR 5%). Dilaporkan perubahan relatif recall dan alarm palsu terhadap test bersih.

In [ ]:
if READY and RUN_ROBUSTNESS:
    rmodel = YOLO(str(FINAL_WEIGHTS))
    LID = [k for k, v in rmodel.names.items() if v == "lying_on_ground"][0]

    def score_arrays(arrs):
        out = []
        for r in rmodel.predict(arrs, conf=SCORE_CONF, imgsz=FINAL_IMGSZ, device=DEVICE, verbose=False):
            b = r.boxes
            if b is None or len(b) == 0:
                out.append(0.0)
                continue
            cls, cf = b.cls.cpu().numpy().astype(int), b.conf.cpu().numpy()
            out.append(float(cf[cls == LID].max()) if (cls == LID).any() else 0.0)
        return np.array(out)

    paired_stems = set(PAIR.stem)
    items_t = [it for it in SPLIT_ITEMS["test"] if it["stem"] in paired_stems]
    chk = score_arrays([cv2.imread(str(it["img"])) for it in items_t[:32]])
    assert np.allclose(chk, PAIR.set_index("stem").loc[[it["stem"] for it in items_t[:32]], "score"].to_numpy(), atol=2e-3), \
        "skor dari array berbeda dengan eval_lying: aturan skor tidak sama"
    t5 = THR[0.05]
    clean = rates_at(S, G, t5)
    rows = [dict(gangguan="bersih", recall=clean["recall"], alarm_palsu=clean["fp_normal"], d_recall_rel=0.0, d_alarm=0)]
    for kind in PERTURBATIONS:
        f = EVAL / f"robust_{kind}.csv"
        if f.exists() and not FORCE_RECOMPUTE:
            sk = pd.read_csv(f)["score"].to_numpy()
        else:
            sk = np.concatenate([score_arrays([perturb(cv2.imread(str(it["img"])), kind, seed=SEED * 100000 + i + j)
                                               for j, it in enumerate(items_t[i:i + 32])])
                                 for i in range(0, len(items_t), 32)])
            pd.DataFrame(dict(stem=[it["stem"] for it in items_t], score=sk)).to_csv(f, index=False)
        gk = PAIR.set_index("stem").loc[[it["stem"] for it in items_t], "gt"].to_numpy()
        rk = rates_at(sk, gk, t5)
        rows.append(dict(gangguan=kind, recall=rk["recall"], alarm_palsu=rk["fp_normal"],
                         d_recall_rel=(rk["recall"] - clean["recall"]) / clean["recall"] if clean["recall"] else np.nan,
                         d_alarm=rk["fp_normal"] - clean["fp_normal"]))
    ROB = pd.DataFrame(rows)
    print(ROB.round(3).to_string(index=False))
    valid = ROB.iloc[1:].dropna(subset=["d_recall_rel"])
    if valid.empty:
        ROB_LABEL = "TIDAK TERDEFINISI"
        print("Perubahan relatif tidak bisa dihitung (recall test bersih = 0); lihat kolom alarm_palsu saja.")
    else:
        worst = valid.sort_values("d_recall_rel").iloc[0]
        print(f"gangguan terburuk untuk recall: {worst.gangguan}")
        ROB_LABEL = show_interp("robust_drop", max(0.0, -float(worst.d_recall_rel)))
    RESULT["robustness"] = ROB.to_dict("records")
elif READY:
    print("RUN_ROBUSTNESS=False: dilewati")

**6g. Kalibrasi skor lying.** Reliability 10 bin dan ECE pada frame lying vs normal. Confidence YOLO bukan probabilitas
terkalibrasi; ECE besar berarti skor tidak boleh dibaca sebagai "peluang jatuh".

In [ ]:
if READY:
    m = np.isin(G, [GT_NORMAL, GT_LYING])
    ece, tab = ece_binary(S[m], (G[m] == GT_LYING).astype(int))
    rel = pd.DataFrame(tab, columns=["bawah", "atas", "n", "rata_conf", "frac_lying"])
    print(rel.round(3).to_string(index=False))
    ECE_LABEL = show_interp("ece", ece)
    ok_ = rel.n > 0
    plt.figure(figsize=(4.6, 4.2))
    plt.plot([0, 1], [0, 1], "--", c="#9a9a96", lw=1)
    plt.plot(rel.rata_conf[ok_], rel.frac_lying[ok_], "o-", c="#2a78d6", lw=2)
    plt.xlabel("rata-rata skor lying (per bin)")
    plt.ylabel("fraksi benar-benar lying")
    plt.title(f"Reliability skor lying (ECE {ece:.3f})")
    plt.show()
    RESULT["ece"] = ece

**6h. Efisiensi.** ms/frame (batch 1, ujung ke ujung), FPS, parameter, GFLOPs, ukuran berkas. Perangkat tujuan belum
diketahui; angka berlaku untuk GPU yang tercetak.

In [ ]:
if READY:
    em = YOLO(str(FINAL_WEIGHTS))
    paths = [str(it["img"]) for it in SPLIT_ITEMS["test"][:110]]
    for p in paths[:10]:
        em.predict(p, imgsz=FINAL_IMGSZ, device=DEVICE, verbose=False)      # pemanasan
    t0 = time.time()
    for p in paths[10:]:
        em.predict(p, imgsz=FINAL_IMGSZ, device=DEVICE, verbose=False)
    ms = 1000 * (time.time() - t0) / max(1, len(paths) - 10)
    size = model_size(FINAL_WEIGHTS)
    print(f"GPU: {GPU_NAME} | batch-1: {ms:.1f} ms/frame (~{1000 / ms:.0f} FPS) | val batch: {MT.speed}")
    print(f"parameter {size['params_M']:.2f} M | {size['gflops']:.1f} GFLOPs | berkas {size['size_MB']:.1f} MB")
    LAT_LABEL = show_interp("latency_ms", ms)
    RESULT["efisiensi"] = dict(gpu=GPU_NAME, ms_batch1=ms, **size)

**6i. Perbandingan berpasangan terhadap baseline v2 dan aturan terima.** Semua komponen dihitung sama untuk kedua model.
Alarm palsu s23 dihitung pada FPR 5% test masing-masing model, karena skor val baseline tidak tersedia.

In [ ]:
if READY:
    PB = paired_bootstrap(ap_fn, SB, S, G, V, N_BOOT, SEED)
    print(f"ΔAP (kandidat - baseline) = {PB['delta']:+.3f}  CI95 [{PB['lo']:+.3f}, {PB['hi']:+.3f}]  "
          f"P(kandidat > baseline) = {PB['p']:.2f}  (bootstrap {PB['n_boot']}x per video)")
    PAIRED_LABEL = show_interp("paired", PB)
    r10_c, r10_b = recall_at_fpr(S, G, 0.10)[0], recall_at_fpr(SB, G, 0.10)[0]

    def s23_fp(sc):
        t = thr_for_fpr(sc[G == GT_NORMAL], 0.05)
        return int(((V == "mcf-s23") & (G == GT_NORMAL) & (sc >= t)).sum())

    S23 = dict(baseline=s23_fp(SB), candidate=s23_fp(S))
    S23_LABEL = show_interp("s23_fa", S23)
    rule = pd.DataFrame([
        dict(syarat="ΔAP > 0 dan P >= 0,9", nilai=f"{PB['delta']:+.3f}, P={PB['p']:.2f}", lolos=PB["delta"] > 0 and PB["p"] >= 0.9),
        dict(syarat="recall@FPR10% tidak turun", nilai=f"{r10_b:.3f} -> {r10_c:.3f}", lolos=r10_c >= r10_b),
        dict(syarat="tidak ada skenario runtuh", nilai=COLLAPSE, lolos=not COLLAPSE.startswith("RUNTUH")),
        dict(syarat="alarm palsu s23 turun", nilai=f"{S23['baseline']} -> {S23['candidate']}", lolos=S23["candidate"] < S23["baseline"]),
    ])
    print(rule.to_string(index=False))
    print("Kandidat " + ("DITERIMA menurut aturan terima" if rule.lolos.all() else "TIDAK diterima menurut aturan terima"))
    print("\nKolom pembanding 'dataset lama' (EXPERIMENTS.md; dataset berbeda, lying test 1 gambar: TIDAK sebanding):")
    print(pd.DataFrame([dict(eksperimen="EXP-006 (dataset lama)", test_mAP50=0.7022),
                        dict(eksperimen="EXP-007 (dataset lama)", test_mAP50=0.5639),
                        dict(eksperimen="kandidat (dataset v2)", test_mAP50=round(float(MT.box.map50), 4))]).to_string(index=False))
    RESULT["paired"] = dict(**PB, recall10_base=r10_b, recall10_cand=r10_c, s23=S23, diterima=bool(rule.lolos.all()))

**6j. Interpretasi otomatis dan verdict.** Ambang = target tim (tabel "Target Metrik" `EXPERIMENTS.md`) atau heuristik
umum. Penjaga n kecil (n video < 10 atau lebar CI > 0,30) mengalahkan label lain. Verdict = yang **terburuk** dari
recall@FPR, skenario runtuh, alarm palsu s23, dan perbandingan berpasangan; nilainya hanya `TIDAK KONKLUSIF`,
`PERLU PERBAIKAN`, atau `KANDIDAT LAYAK DIUJI LANJUT`. Notebook ini tidak menyimpulkan "layak deploy": metrik per frame
dari 6 video lying belum sama dengan kinerja per kejadian (>10 detik), dan ambang backend serta perangkat tujuan belum
dikonfirmasi.

In [ ]:
if READY:
    t5 = THR[0.05]
    rec_wp, lo, hi = bootstrap_by_video(lambda s, g: rates_at(s, g, t5)["recall"], S, G, V, N_BOOT, SEED)
    prec_wp, plo, phi = bootstrap_by_video(lambda s, g: rates_at(s, g, t5)["precision"], S, G, V, N_BOOT, SEED)
    print("Titik kerja val FPR5%:")
    REC_LABEL = show_interp("recall_lying", rec_wp, n_videos=N_VID_LYING, ci=(lo, hi))
    show_interp("precision", prec_wp, n_videos=N_VID_LYING, ci=(plo, phi))
    r5, r5lo, r5hi = bootstrap_by_video(make_recall_fpr_fn(0.05), S, G, V, N_BOOT, SEED)
    print("Recall@FPR5% (deskriptif test):")
    show_interp("recall_lying", r5, n_videos=N_VID_LYING, ci=(r5lo, r5hi))
    show_interp("map50", float(MT.box.map50))
    components = {"recall_lying": REC_LABEL, "collapse": COLLAPSE, "s23_fa": S23_LABEL, "paired": PAIRED_LABEL}
    VERDICT = final_verdict(components)
    print("\nKomponen verdict:", components)
    print(f"\n>>> VERDICT: {VERDICT}  ({WARN_STD})")
    print(f"Bukan keputusan deploy: metrik per frame dari {N_VID_LYING} video lying; ambang backend dan perangkat tujuan belum dikonfirmasi.")
    RESULT.update(verdict=VERDICT, components=components, weights=str(FINAL_WEIGHTS), scheme=FINAL_SCHEME,
                  candidate=CANDIDATE, seeds=SEED_ROWS, n_video_lying_test=N_VID_LYING)
    (EVAL / "summary.json").write_text(json.dumps(RESULT, indent=2, default=str))
    print(f"Ringkasan untuk draf entri eksperimen: {EVAL / 'summary.json'}")

## Catatan dan langkah berikutnya
- **Audit pseudo-keypoint** (`WORK/diag/pseudo_kp_lying.png`) harus dinilai Laila sebelum hasil pose dibahas. Bila
  keypoint pada frame lying banyak yang salah, kepala pose belajar dari label yang keliru.
- **Hipotesis "lying = tubuh rebah di atas alas"** (±87% lying latih berada di atas matras) tidak diuji langsung oleh
  notebook ini. Tanda pendukung: alarm palsu yang tetap terkonsentrasi di s23 (sofa) dan runtuhnya s19/s22.
- **Sequence/temporal** tidak dibangun: model satu frame. Smoothing N sampel dan aturan durasi ada di backend/EXP-010.
- Draf entri eksperimen ditulis terpisah (`docs/draft_exp_entry.md`); `EXPERIMENTS.md` hanya diubah setelah tim
  menyetujui nomor eksperimennya.